## Test Open SANOS



In [ ]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.sampler
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.torch
%aimport quantdata.torch.bbtorch
%aimport quantdata.torch.ffw
%aimport quantdata.dysanos.simple
%aimport quantdata.dysanos.simple.sampler
%aimport quantdata.dysanos.simple.model
%aimport quantdata.sanos.utils
%aimport quantdata.sanos.open
%aimport quantdata.sanos.fitter

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"]).data['SPX']
print(f"/done; columns {data.keys()}")


In [ ]:
%aimport quantdata.sanos.utils
%aimport quantdata.sanos.open
%aimport quantdata.sanos.fitter

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store
from cdxcore.verbose import Context
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.sanos.open import OpenGeometryGenerator, OpenSanos
from quantdata.sanos.fitter import Eps, Fitter
import numpy as np
from cdxcore.pretty import PrettyObject

def fit(data):
    num_samples = len(data.dates)

    # configure open geometry generation
    # ----------------------------------

    geometry_generator = OpenGeometryGenerator(
                            dlv_min     = 0.,
                            dlv_max     = 10.,
                            fwdvol_min  = 0.0,
                            vol_interp  = "lin_var",
                            min_dx      = 0.001,
                            max_dx      = 0.05,
                            lower_scale = 0.8,
                            upper_scale = 1.2,
                            max_dx_outer_fac = 2.
                        )

    eps         = Eps()

    def fit_sample( isample, store, min_num_strikes = 10):

        full_market = PrettyObject()
        is_valid    = data.is_valid[isample,:]
        d           = data.dates[isample]
        full_market.is_am      = data.is_am[isample,is_valid]
        full_market.strikes    = data.pure_strike[isample,is_valid]
        full_market.tte_y      = data.tte_y[isample,is_valid]
        full_market.bid_vol    = data.bid_vol[isample,is_valid]
        full_market.ask_vol    = data.ask_vol[isample,is_valid]
        full_market.bid        = data.bid_pure_price[isample,is_valid]
        full_market.ask        = data.ask_pure_price[isample,is_valid]

        # construct open geometry
        # -----------------------

        reduced_market = PrettyObject()
        for k in full_market.keys():
            reduced_market[k] = []

        src_expiries       = np.unique(full_market.tte_y)
        market_strikes    = []
        market_expiries   = []
        for expiry in src_expiries:
            mask      = full_market.tte_y == expiry
            strikes_t = full_market.strikes[mask]

            if len(strikes_t) >= min_num_strikes:
                market_strikes.append(np.sort(strikes_t))
                market_expiries.append(expiry)

                for k in full_market.keys():
                    reduced_market[k].append(full_market[k][mask])

        reduced_market  = PrettyObject({ k: np.concatenate(v) for k,v in reduced_market.items() })
        market_expiries = np.asarray(market_expiries, dtype=np.float64)
        geometry        = geometry_generator.create_std_geometry(
                            expiries     = market_expiries,
                            pure_strikes = market_strikes,
                            dtype        = np.float64,
                      )

        fitter      = Fitter( geometry = geometry, eps = eps, debug=True, max_error = None, mid_fit_weight = 0.0001 )
        sanos       = OpenSanos( geometry = geometry, eps = eps, debug=True )

        # plot the geometry
        # -----------------

        with figure(f"Open Geometry {d}", col_size=5, row_size=4, columns=None ) as fig_strikes:
            axD, axA    = fig_strikes.add_subplots("Inner", "All")
            axD.set_xlabel("TTE")
            axA.set_xlabel("TTE")
            axD.set_ylabel("Pure Strike (log-scale)")
            axA.set_ylabel("Pure Strike (log-scale)")
            axA.set_yscale("log")
            axD.set_yscale("log")

            for xexpiry, xstrikes_t in zip(geometry.xexpiries, geometry.xstrikes):
                texp_all   = np.full(xstrikes_t.shape, xexpiry, dtype=np.float64)
                texp_inner = np.full(xstrikes_t[1:-1].shape, xexpiry, dtype=np.float64)
                axA.plot( texp_all, xstrikes_t, ".:" )
                axD.plot( texp_inner, xstrikes_t[1:-1], ".:" )

            fig_strikes.render()
            store = fig_strikes.store()

            def plot_strikes(store):
                utty, inv = np.unique(reduced_market.tte_y, return_inverse=True)

                min_strike = np.full(len(utty), np.inf, dtype=reduced_market.strikes.dtype)
                max_strike = np.full(len(utty), -np.inf, dtype=reduced_market.strikes.dtype)

                np.minimum.at(min_strike, inv, reduced_market.strikes)
                np.maximum.at(max_strike, inv, reduced_market.strikes)
            
                store.clear()
                store += axA.fill_between(utty, min_strike, max_strike, alpha=0.3, color="blue")
                store += axD.fill_between(utty, min_strike, max_strike, alpha=0.3, color="blue")            
                store += axD.plot(utty, min_strike, ".-", color="blue")
                store += axD.plot(utty, max_strike, ".-", color="blue")
                store += axD.plot(reduced_market.tte_y, reduced_market.strikes, ".", color="blue")
                fig_strikes.render()

                return utty, min_strike, max_strike

            plot_strikes(store)


        # compute prices, dlvs
        # --------------------

        FWD_VOLS       = 0.05

        dlvsqs, fit_prices, densities = \
            fitter( fwd_vols     = FWD_VOLS,
                    opt_strikes  = reduced_market.strikes,
                    opt_bid      = reduced_market.bid,
                    opt_ask      = reduced_market.ask,
                    opt_tte      = reduced_market.tte_y,
                    opt_is_calls = None,
                    opt_mask     = None,
                    return_only_dlvsqs = False,
                    verbose      = Context.all ) 
        assert fit_prices.shape == reduced_market.strikes.shape, f"Expected fit_prices to have shape {reduced_market.strikes.shape}, got {fit_prices.shape}"
        fit_vols  = bs.implied(reduced_market.strikes, fit_prices, sqrtT = np.sqrt(reduced_market.tte_y) )

        re_prices = sanos.price(  fwd_vols     = FWD_VOLS,
                                  densities    = densities,
                                  opt_strikes  = full_market.strikes,
                                  opt_tte      = full_market.tte_y
                               )
        re_vols  = bs.implied(full_market.strikes, re_prices, sqrtT = np.sqrt(full_market.tte_y) )

        # DLVs, manual
        # ------------

        with figure(f"Fits {d}", col_size=5, row_size=4, columns=4 ) as fig_fits:

            uttys = np.unique(full_market.tte_y)

            for utty in uttys:
                full_mask = full_market.tte_y == utty
                ax = fig_fits.add_subplot(f"TTE {utty*DAYS_PER_YEAR:.3f}d {full_market.is_am[full_mask][0]}")
                ax.set_xlabel("Pure Strike")
                ax.set_ylabel("Implied")

                istrike = full_market.strikes[full_mask]
                ixs     = np.argsort(istrike)

                ax.plot(istrike[ixs], full_market.bid_vol[full_mask][ixs], "^-", color="green", label="Bid")
                ax.plot(istrike[ixs], full_market.ask_vol[full_mask][ixs], "v-", color="red", label="Ask")
                ax.plot(istrike[ixs], re_vols[full_mask][ixs], ".-", color="purple", label="Repriced")

                if utty in reduced_market.tte_y:
                    red_mask = reduced_market.tte_y == utty
                    irstrike = reduced_market.strikes[red_mask]
                    ixs      = np.argsort(irstrike)
                    ax.plot(irstrike[ixs], fit_vols[red_mask][ixs], "*:", color="orange", label="Fit")
                ax.legend(loc="upper left")

    for isample in range(num_samples-1,num_samples-200,-20):
        fit_sample(isample, store)
        break

fit(data)
